# Embed the Statements and Take the Centroid

| Item | Description |
|---|---|
| Goal | Embed every Topic 68 stance statement and take their centroid: topic vector c, Option 2. |
| Input | `output/T3.5.2/topic68_statements.csv`, one row per statement with its text. |
| Output | `output/T3.5.3/statement_embeddings.npy` (374 × 1024, float32, one row per statement in the input order), `output/T3.5.3/topic_centroid_embedding.npy` (1 × 1024, float32), and `output/T3.5.3/nearest_statements.csv` (the ten statements closest to the centroid). |
| Unit | A statement, as defined in T3.5.2. |
| Model | `BAAI/bge-large-en-v1.5` from the project's local snapshot: the CLS token of the last layer, L2-normalized, at most 512 input tokens. Run on the GPU in float32. |
| Variables | • `statement_embeddings`: $e_j$, the unit-length embedding of statement $j$.<br>• `topic_centroid_embedding`: $c_{\text{cent}} = \bar{e} / \lVert \bar{e} \rVert$ with $\bar{e} = \frac{1}{J}\sum_{j=1}^{J} e_j$, the mean of the $J = 374$ statement embeddings scaled to unit length.<br>• `similarity_to_centroid`: $\cos(e_j, c_{\text{cent}}) = e_j \cdot c_{\text{cent}}$. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for how this vector is used downstream.


## Step 1. Import packages

Read the statement table, load the encoder, and save the vectors.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from IPython.display import display


## Step 2. Define file locations

Read the statement table and the model snapshot in place; save outputs in `output/T3.5.3/` under the TEMP folder.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.5.3"  # the identifier names this notebook's output folder
STATEMENTS_FILE = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output/T3.5.2/topic68_statements.csv"
MODEL_FOLDER = Path(
    "/scratch2/acroh/projects/llm_method/hf_cache/hub/models--BAAI--bge-large-en-v1.5"
    "/snapshots/d4aa6901d3a41ba39fb536a557fa166f842b0e09"
)
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME

MAX_TOKENS = 512  # the encoder's input limit
BATCH_SIZE = 64  # statements encoded per forward pass
N_NEAREST = 10  # statements listed as the readability check
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Outputs: {OUTPUT_FOLDER}")


Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.3


## Step 3. Load the statements

Keep the input order; the embedding matrix rows follow it. Every statement must fit the encoder's input limit.


In [3]:
statements = pd.read_csv(STATEMENTS_FILE)
assert statements["text"].notna().all()
assert statements["n_tokens"].le(MAX_TOKENS).all()
assert not statements.duplicated(["packageId", "turn_idx", "record_idx"]).any()
print(f"Statements: {len(statements):,}; by stance: {statements['stance'].value_counts().to_dict()}")
display(statements[["packageId", "turn_idx", "record_idx", "hearing_year", "stance", "n_tokens"]].head(3))


Statements: 374; by stance: {'support': 221, 'concern': 153}


,packageId,turn_idx,record_idx,hearing_year,stance,n_tokens
0,CHRG-106shrg69984,4,1,1999,support,39
1,CHRG-106shrg69984,9,7,1999,support,48
2,CHRG-106shrg69984,9,8,1999,support,48


## Step 4. Load the encoder

Load the tokenizer and model from the local snapshot only, in evaluation mode on the GPU, and check the 512-token limit.


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_FOLDER, local_files_only=True)
assert torch.cuda.is_available(), "run this notebook on a GPU node"
device = torch.device("cuda")
model = AutoModel.from_pretrained(MODEL_FOLDER, local_files_only=True).to(device).eval()
assert model.config.max_position_embeddings == MAX_TOKENS
print(f"Encoder: {MODEL_FOLDER.parent.parent.name}; hidden size {model.config.hidden_size}; input limit {MAX_TOKENS} tokens")
print(f"Device: {torch.cuda.get_device_name(device)}; dtype {next(model.parameters()).dtype}")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Encoder: models--BAAI--bge-large-en-v1.5; hidden size 1024; input limit 512 tokens
Device: NVIDIA A40; dtype torch.float32


## Step 5. Embed every statement

Encode in batches with padding, take the CLS token of the last layer for each statement, and scale each vector to unit length. Padding does not change a statement's vector because the attention mask excludes the padded positions.


In [5]:
texts = statements["text"].tolist()
statement_embeddings = np.empty((len(texts), model.config.hidden_size), dtype=np.float32)
with torch.no_grad():
    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start + BATCH_SIZE]
        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=MAX_TOKENS, return_tensors="pt"
        ).to(device)
        last_layer = model(**encoded).last_hidden_state
        cls_vectors = torch.nn.functional.normalize(last_layer[:, 0], dim=-1)
        statement_embeddings[start:start + len(batch)] = cls_vectors.cpu().numpy()
assert statement_embeddings.shape == (len(statements), model.config.hidden_size)
assert np.allclose(np.linalg.norm(statement_embeddings, axis=1), 1, atol=1e-5)
print(f"Embedded {len(statements):,} statements; matrix {statement_embeddings.shape}, {statement_embeddings.dtype}")


Embedded 374 statements; matrix (374, 1024), float32


## Step 6. Take the centroid

The centroid is the mean of the statement vectors scaled back to unit length. The mean's length before scaling says how tightly the statements cluster: 1 would mean identical vectors, 0 no common direction.


In [6]:
mean_vector = statement_embeddings.mean(axis=0)
mean_length = float(np.linalg.norm(mean_vector))
topic_centroid_embedding = (mean_vector / mean_length).astype(np.float32).reshape(1, -1)
assert abs(np.linalg.norm(topic_centroid_embedding) - 1) < 1e-5
print(f"Mean vector length before scaling: {mean_length:.4f}")
print(f"Centroid shape: {topic_centroid_embedding.shape}; first five values: {np.round(topic_centroid_embedding[0, :5], 4)}")


Mean vector length before scaling: 0.7504
Centroid shape: (1, 1024); first five values: [-0.0061 -0.0295 -0.0115  0.0407  0.0114]


## Step 7. List the statements nearest the centroid

Cosine similarity of each statement to the centroid is the dot product of unit vectors. The ten nearest statements show, in plain words, what the centroid stands for.


In [7]:
statements["similarity_to_centroid"] = statement_embeddings @ topic_centroid_embedding[0]
print(f"Similarity to the centroid: min {statements['similarity_to_centroid'].min():.3f}, median {statements['similarity_to_centroid'].median():.3f}, max {statements['similarity_to_centroid'].max():.3f}")
nearest_statements = statements.nlargest(N_NEAREST, "similarity_to_centroid")[
    ["similarity_to_centroid", "stance", "hearing_year", "packageId", "turn_idx", "record_idx", "text"]
]
pd.set_option("display.max_colwidth", 200)
display(nearest_statements)


Similarity to the centroid: min 0.484, median 0.759, max 0.879


,similarity_to_centroid,stance,hearing_year,packageId,turn_idx,record_idx,text
97,0.878806,concern,2008,CHRG-110shrg76329,1,1,"I think that the issue here of privacy, who is collecting what kind of information, how does that information exist in what identifiable form or how might that be used, who is it sold to, does the..."
271,0.876010,support,2019,CHRG-116shrg60880,113,0,"Well, I happen to believe that this is one of the areas, maybe not many areas in this next couple of years, that we ought to be able to come together around in a bipartisan way and come up with a ..."
364,0.865383,concern,2024,CHRG-118shrg61871,4,0,"As I conclude, I would like to address a related matter, the American Privacy Rights Act--APRA. I support Congress--not the FTC or any Federal agency but Congress setting a nationwide data privacy..."
14,0.861175,concern,2000,CHRG-106shrg81862,17,0,All of us are concerned about consumer privacy. I am concerned that consumers who want privacy should have privacy.
72,0.856054,concern,2003,CHRG-108shrg81138,140,0,There is a strong concern about the privacy issues involved here.
182,0.853792,support,2016,CHRG-114shrg25081,336,0,Senators Booker and Blumenthal and I have written to the Commission urging the Commission to initiate a rulemaking to protect the privacy of consumers who use the broadband. And I do believe you s...
79,0.853207,support,2004,CHRG-108shrg20672,115,0,"I won't belabor this, but other than the definitions about explicit permission, that sentence I read sure sounds like the predicate for a general online privacy bill, which takes us back to Burns/..."
52,0.851989,support,2001,CHRG-107shrg88997,12,2,"My message to folks in the private sector is that if they want some measure of preemption, they have got to support a bill with meaningful privacy protection. There has got to be meaningful privac..."
312,0.850992,support,2021,CHRG-117shrg53125,4,0,I think it is very important to understand that our committee would like to move forward on stronger privacy legislation. And yesterday's hearing clearly crystallized that we need to update the Ch...
34,0.848143,support,2000,CHRG-106shrg85657,3,2,That is why myself and the other cosponsors have introduced our bill after a complete study and 5 years of the FTC trying to get self-regulation. There is no doubt in a comprehensive field as the ...


## Step 8. Save the vectors and the check

Save the statement matrix in the input order, the centroid, and the nearest-statement table.


In [8]:
embeddings_file = OUTPUT_FOLDER / "statement_embeddings.npy"
centroid_file = OUTPUT_FOLDER / "topic_centroid_embedding.npy"
nearest_file = OUTPUT_FOLDER / "nearest_statements.csv"
np.save(embeddings_file, statement_embeddings)
np.save(centroid_file, topic_centroid_embedding)
nearest_statements.to_csv(nearest_file, index=False)
print(f"Saved: {embeddings_file}")
print(f"Saved: {centroid_file}")
print(f"Saved: {nearest_file}")


Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.3/statement_embeddings.npy
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.3/topic_centroid_embedding.npy
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.3/nearest_statements.csv
